In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import re
from datetime import datetime

In [0]:
# Initialize Spark Session
spark = SparkSession.builder \
    .appName("LA_Crime_Bronze_Silver_Pipeline") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

# Set database paths
BRONZE_PATH = "/Volumes/workspace/damg7370/datastore/LA CRIME  DATA/bronze/"
SILVER_PATH = "/Volumes/workspace/damg7370/datastore/LA CRIME  DATA/silver/"
VOLUME_PATH = "/Volumes/workspace/damg7370/datastore/LA CRIME  DATA/la_crime_clean.csv"  # Adjust based on your Databricks setup

In [0]:
def sanitize_column_names(df):
    for col in df.columns:
        new_col = re.sub(r'[ ,;{}()\n\t=]', '_', col)
        df = df.withColumnRenamed(col, new_col)
    return df

@dlt.table(
    name="la_crime_raw",
    comment="Bronze table for LA crime data"
)
def bronze_la_crime():
    file_path = VOLUME_PATH
    df_bronze = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .option("multiLine", "true")
        .option("escape", '"')
        .csv(file_path)
    )
    df_bronze = sanitize_column_names(df_bronze)
    df_bronze = (
        df_bronze
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file", lit(file_path))
        .withColumn("bronze_record_id", monotonically_increasing_id())
    )
    return df_bronze

In [0]:

@dlt.view(
    name="profile_la_crime_raw"
)
def profile_bronze_data():
    df = dlt.read("la_crime_raw")
    
    print("=" * 80)
    print("DATA PROFILING REPORT")
    print("=" * 80)
    print("\n SCHEMA INFORMATION:")
    df.printSchema()
    
    total_records = df.count()
    print(f"\n📈 TOTAL RECORDS: {total_records:,}")
    
    print("\n🔍 NULL VALUE ANALYSIS:")
    null_counts = df.select([
        count(
            when(
                col(c).isNull() | (col(c) == ""), c
            )
        ).alias(c)
        for c in df.columns if c not in ['ingestion_timestamp', 'source_file', 'bronze_record_id']
    ])
    null_df = null_counts.collect()[0].asDict()
    for col_name, null_count in null_df.items():
        if total_records > 0:
            null_pct = (null_count / total_records) * 100
            print(f"   {col_name}: {null_count:,} nulls ({null_pct:.2f}%)")
        else:
            print(f"   {col_name}: {null_count:,} nulls (N/A%)")
    
    print("\n DATE RANGE ANALYSIS:")
    if total_records > 0:
        date_stats = df.select(
            min(col("DATE_OCC")).alias("min_date"),
            max(col("DATE_OCC")).alias("max_date"),
            countDistinct(col("DATE_OCC")).alias("distinct_dates")
        ).collect()[0]
        print(f"   Date Range: {date_stats['min_date']} to {date_stats['max_date']}")
        print(f"   Distinct Dates: {date_stats['distinct_dates']:,}")
    else:
        print("   Date Range: N/A")
        print("   Distinct Dates: N/A")
    
    print("\n KEY COLUMNS DISTINCT COUNTS:")
    key_columns = ["DR_NO", "AREA", "Crm Cd", "Vict Sex", "Status", "Weapon Used Cd"]
    for col_name in key_columns:
        if col_name in df.columns:
            if total_records > 0:
                distinct_count = df.select(col_name).distinct().count()
                print(f"   {col_name}: {distinct_count:,} distinct values")
            else:
                print(f"   {col_name}: N/A distinct values")
    
    return df